Hybrid Search RAG Using Pinecone

In [1]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

#set up environment
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
api_key=os.getenv("PINECONE_API_KEY")

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec

index_name="hybrid-search-langchain-pinecone"

#intilize pinecone client
pc=Pinecone(api_key=api_key)

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #dimensone of dense vector ->it is 384 we use huggingface embedding which default converts any text in 384 dimenson vector
        metric='dotproduct',
        spec=ServerlessSpec(cloud='aws',region='us-east-1')

    )

ImportError: cannot import name 'Pinecone' from 'pinecone' (unknown location)

In [46]:
index=pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-xnexi8r.svc.aped-4627-b74a.pinecone.io')

In [47]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14279.54it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [48]:
from pinecone_text.sparse import BM25Encoder
#this encode uses TF-IDF technique by default

bm25_encoder=BM25Encoder().default()
bm25_encoder


In [49]:
sentences=[
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

bm25_encoder.fit(sentences)

#store values in to json file
bm25_encoder.dump("bm25_values.json")

bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<?, ?it/s]


In [50]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001FC1713C950>, index=Index(host='https://hybrid-search-langchain-pinecone-xnexi8r.svc.aped-4627-b74a.pinecone.io'))

In [53]:
import inspect
from langchain_community.retrievers import pinecone_hybrid_search

print(inspect.getsource(pinecone_hybrid_search.create_index))

def create_index(
    contexts: List[str],
    index: Any,
    embeddings: Embeddings,
    sparse_encoder: Any,
    ids: Optional[List[str]] = None,
    metadatas: Optional[List[dict]] = None,
    namespace: Optional[str] = None,
    text_key: str = "context",
) -> None:
    """Create an index from a list of contexts.

    It modifies the index argument in-place!

    Args:
        contexts: List of contexts to embed.
        index: Index to use.
        embeddings: Embeddings model to use.
        sparse_encoder: Sparse encoder to use.
        ids: List of ids to use for the documents.
        metadatas: List of metadata to use for the documents.
        namespace: Namespace value for index partition.
    """
    batch_size = 32
    _iterator = range(0, len(contexts), batch_size)
    try:
        from tqdm.auto import tqdm

        _iterator = tqdm(_iterator)
    except ImportError:
        pass

    if ids is None:
        # create unique ids using hash of the text
        ids = [has

In [54]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]
)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: Index.upsert() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given